In [ ]:
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from scipy import stats
import numpy as np
from sklearn.metrics import mean_squared_error

In [ ]:
crime = pd.read_csv("crime-housing-austin-2015.csv")

# Figure out best geographical unit to divide statistics

In [ ]:
crime.groupby('Zip_Code_Crime').Medianhomevalue.unique()
crime.groupby('Zip_Code_Crime').Unemployment.unique()
crime.groupby('Zip_Code_Crime').Rentalunitsaffordabletoaverageartist.unique()

In [ ]:
crime.groupby('Census_Tract').Medianhomevalue.unique()
crime.groupby('Census_Tract').Unemployment.unique()
crime.groupby('Census_Tract').Rentalunitsaffordabletoaverageartist.unique()

In [ ]:
crime.groupby('Council_District').Medianhomevalue.unique()
crime.groupby('Council_District').Unemployment.unique()
crime.groupby('Council_District').Rentalunitsaffordabletoaverageartist.unique()

In [ ]:
crime.groupby('District').Medianhomevalue.unique()
crime.groupby('District').Unemployment.unique()
crime.groupby('District').Rentalunitsaffordabletoaverageartist.unique()

Figured out that the statistics are calculated using Zip Code

# Violent vs Nonviolent Crime

In [ ]:
crime.Highest_Offense_Desc.unique()

In [ ]:
crime['violent_crime'] = crime.Highest_Offense_Desc.isin(['AGG ROBBERY/DEADLY WEAPON','ROBBERY BY ASSAULT','AGG ASLT W/MOTOR VEH FAM/DAT V',
    'AGG ASLT STRANGLE/SUFFOCATE','AGG ASSAULT','AGG ASLT ENHANC STRANGL/SUFFOC','RAPE','DEADLY CONDUCT','AGG ASSAULT FAM/DATE VIOLENCE',
    'AGG RAPE OF A CHILD','AGG RAPE','ROBBERY BY THREAT','AGG ROBBERY BY ASSAULT','RAPE OF A CHILD','AGG ASSAULT WITH MOTOR VEH','MURDER',
    'AGG ASSAULT ON PUBLIC SERVANT','BURG OF RES - SEXUAL NATURE', 'DEADLY CONDUCT FAM/DATE VIOL','TAKE WEAPON FRM POLICE OFFICER',
    'MANSLAUGHTER'])

crime['violent_crime_string'] = crime['violent_crime'].astype('string')

ax = sns.countplot(x="violent_crime_string", data=crime)
ax.set_xticklabels(["Violent Crimes", "Nonviolent Crimes"])
ax.set_ylabel("Count")
ax.set_title("Number of Violent Crimes vs Nonviolent Crimes\nin Austin, Texas (2015)")
ax.set_xlabel("Crime Type")

plt.tight_layout()
plt.show()

In [ ]:
crime_by_zip = crime.groupby('Zip_Code_Crime')['violent_crime_string'].value_counts().unstack(fill_value=0)

ax = sns.scatterplot(x = 'False', y = 'True', data = crime_by_zip)

ax.set_title("Nonviolent vs Violent Crimes by Zip Code\nin Austin, Texas (2015)")
ax.set_xlabel("Nonviolent Crimes")
ax.set_ylabel("Violent Crimes")

plt.tight_layout()
plt.show()

In [ ]:
stats.pearsonr(crime_by_zip['False'], crime_by_zip['True'])

In [ ]:
sns.regplot(x = 'False', y = 'True', data = crime_by_zip,
    order=1,
    line_kws={"color": "red", "label": "Straight Line"}
)

sns.regplot(x = 'False', y = 'True', data = crime_by_zip,
    order=2,
    scatter_kws={"color": "blue"},
    line_kws={"color": "green", "label": "Quadratic Line"},
)

plt.title("Nonviolent vs Violent Crimes by Zip Code\nin Austin, Texas (2015)")
plt.xlabel("Nonviolent Crimes")
plt.ylabel("Violent Crimes")
plt.legend()

plt.tight_layout
plt.show()

coef_lin = np.polyfit(crime_by_zip["False"], crime_by_zip["True"], 1)
pred_lin = np.polyval(coef_lin, crime_by_zip["False"])
rmse_lin = np.sqrt(mean_squared_error(crime_by_zip["True"], pred_lin))


coef_quad = np.polyfit(crime_by_zip["False"], crime_by_zip["True"], 2)
pred_quad = np.polyval(coef_quad, crime_by_zip["False"])
rmse_quad = np.sqrt(mean_squared_error(crime_by_zip["True"], pred_quad))

print(f"Straight Line RMSE: {rmse_lin:.4f}")
print(f"Quadratic Line RMSE: {rmse_quad:.4f}")


The quadratic line fits slightly better, and the line shows that zip codes with more crime tend to have a higher ratio of violent crime. This effect isn't very large, but it does have theoretical backing.

# Unemployment

In [ ]:
population = pd.read_csv("AustinZipCodes.csv")

In [ ]:
crime['unemployment_num'] = pd.to_numeric(crime.Unemployment.str.replace("%", "", regex=False))
sns.displot(crime.unemployment_num)

In [ ]:
crime_pop = crime.merge(population, left_on='Zip_Code_Crime', right_on='Zip Code')
crime_pop['Population'] = pd.to_numeric(crime_pop['Population'].str.replace(",", "", regex=False))
crime_num_zip = crime_pop.groupby('Zip_Code_Crime').size().to_frame().rename(columns={0: "num_crimes"})
crime_pop_zip = crime_pop.groupby('Zip_Code_Crime').agg({"unemployment_num": ["mean"], "Population": ["mean"]}).droplevel(level=1, axis=1)
crime_per_capita = crime_num_zip.merge(crime_pop_zip, on = 'Zip_Code_Crime')

crime_per_capita['unemployment_level'] = crime_per_capita['unemployment_num'].case_when(
    [
        (crime_per_capita['unemployment_num'] > 12, 'High'),
        (crime_per_capita['unemployment_num'] > 8, 'Medium High'),
        (crime_per_capita['unemployment_num'] > 4, 'Medium Low'),
        (crime_per_capita['unemployment_num'] >= 0, 'Low'),
    ]
)

In [ ]:
crime_per_capita = crime_per_capita.groupby('unemployment_level').agg(
    {"num_crimes": ["sum"], "Population": ["sum"]}
    ).reset_index().droplevel(level=1, axis=1)
crime_per_capita['crimes_per_capita'] = crime_per_capita.num_crimes / crime_per_capita.Population

plt.figure(figsize=(9.5, 7))
ax = sns.barplot(data = crime_per_capita, x = 'unemployment_level', y = 'crimes_per_capita',
            order = ['Low', 'Medium Low', 'Medium High', 'High'])

ax.set_xticklabels(['Low (0-4%)', 'Moderately Low (5-8%)', 'Moderately High (9-12%)', 'High (13-16%)'])

plt.title("Crimes per Capita by Unemployment Level in Austin, Texas (2015)")
plt.xlabel("Unemployment Level")
plt.ylabel("Crimes per Capita")

plt.show()